# **TUTORIAL**: Study objective B1 (deterministic)

AESA Phase B: allocated shares of carrying capacities.\
Compute **deterministic** allocated shares of carrying capacities (aSoCC). Workflows can include <span style="color:#366e9c"><strong>py</strong></span><span style="color:#c83737"><strong>aesa</strong></span> computed aSoCC methods and staged external user
provided aSoCC methods.

# Before starting...

### Set workspace

Run `set_workspace(...)` before any later function call. It sets the active workspace root
for the current Python session, creates or reuses the workspace, imports package
prerequisites under `data_raw/`, and records setup guidance in `data_raw/summary.log`.

In [ ]:
from pyaesa import set_workspace

# Windows example; update this path before running.
set_workspace(r"C:\Users\username\Documents\aesa_workspace")

# macOS example; update this path before running.
# set_workspace("/Users/username/Documents/aesa_workspace")

### Run prerequisites

This tutorial assumes that the workspace has already been set, with all prerequisites available (downloads and processing).\
If this is not your case, it is recommended to first go through the core prerequisite notebooks: [tutorials/core_prerequisites/0_set_workspace.ipynb](../../core_prerequisites/0_set_workspace.ipynb), [tutorials/core_prerequisites/1_download_data.ipynb](../../core_prerequisites/1_download_data.ipynb), and [tutorials/core_prerequisites/2_process_data.ipynb](../../core_prerequisites/2_process_data.ipynb).

### Functional units and selectors

The example in this tutorial uses the **functional unit** `L2.c.b` with the following **selectors**: one producing sector `s_p="Paper"` and one consuming region `r_c="FR"`.

Use [tutorials/study_objectives/1_functional_units_and_allocation_methods.md](../1_functional_units_and_allocation_methods.md) to choose the FU and required selectors. Use <a href="../../methodological_notes/methodological_note__asocc_fus_allocation_methods.pdf">methodological_notes/methodological_note__asocc_fus_allocation_methods.pdf</a> for detailed FU definitions and mathematical expressions.

In [ ]:
project_name_ = "SO_B1_demo_paper_fr"

source_ = "exiobase_3102_ixi"
fu_code_ = "L2.c.b"
study_period_ = range(2010, 2021)
s_p_ = ["Paper"]
r_c_ = ["FR"]
lcia_method_ = "pb_lcia"

# Basic features of the deterministic function

### In a nutshell

The function takes necessary **inputs**, mainly for three purposes:
- public scope arguments, such as `project_name`, `source`, `years`, `lcia_method`, `fu_code` and the relevant selectors (here `s_p` and `r_c`) for the selected functional unit.
- allocation method arguments, such as `method_plan`, `l1_methods`, `one_step_methods`, `two_step_methods`, `l1_l2_pairs`, and `l1_reg_aggreg`.
- enacting metric settings, such as `reference_years`, `ssp_scenario`, `projection_mode`, `reg_window`, and `l2_reuse_years`.

The deterministic **output** folder contains:
- allocated share tables for the resolved L1 or L2 scope. Enacting metrics used to compute the allocated shares are optionally written with `intermediate_outputs`.
- **Figures** are rendered by default (`figures=True`). Use `figures=False` to skip them; `figure_format` controls the file format and resolution. `figure_options` can restrict figure families.
- log files

Basic features also involve:
- **MRIO aggregation and disaggregation** of sectors and/or regions: use the `agg_sec`, `agg_reg`, and `agg_version` parameters. The MRIO aggregation folder includes packaged `agg_reg_eu27.csv` and `agg_reg_world.csv` examples; inspect them in `data_raw/mrio/<source>/aggregation/` and follow `README_aggregation.txt` before writing a custom MRIO aggregation and disaggregation mapping CSV.
- **allocation methods**: by default, <span style="color:#366e9c"><strong>py</strong></span><span style="color:#c83737"><strong>aesa</strong></span> runs all allocation methods available for the selected FU. If a study uses a smaller method set, justify that normative choice and use `method_plan`, `l1_methods`, `one_step_methods`, `two_step_methods`, and `l1_l2_pairs`. Available methods per FU are listed in [tutorials/study_objectives/1_functional_units_and_allocation_methods.md](../1_functional_units_and_allocation_methods.md); definitions and mathematical expressions are in <a href="../../methodological_notes/methodological_note__asocc_fus_allocation_methods.pdf">methodological_notes/methodological_note__asocc_fus_allocation_methods.pdf</a>.
- **overwriting** of existing values: use the `refresh` parameter.

### Public argument checklist
The table lists all arguments; the same definitions are available in the function docstring.

<div class="pyaesa-argument-legend">
<div class="pyaesa-default-block" style="color:#087f5b"><strong>Green items = default if omitted.</strong></div>
<div class="pyaesa-optional-block" style="color:#c45f00"><strong>Orange items = optional feature skipped if omitted.</strong></div>
</div>

Do not write green or orange items when that behavior is intended.

<details open>
<summary><code>deterministic_asocc(...)</code> arguments</summary>

<table>
<thead><tr><th>Argument</th><th>Description</th></tr></thead>
<tbody>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>project_name</code></td><td>Required project name used to build<br>&#10;<code>&lt;repo&gt;/&lt;project_name&gt;</code>.</td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>source</code></td><td>MRIO source key (<code>&quot;exiobase_396_ixi&quot;</code>,<br>&#10;<code>&quot;exiobase_396_pxp&quot;</code>, <code>&quot;exiobase_3102_ixi&quot;</code>,<br>&#10;<code>&quot;exiobase_3102_pxp&quot;</code>, or <code>&quot;oecd_v2025&quot;</code>), or <code>&quot;iso3&quot;</code><br>&#10;for ISO3 only mode (L1 EG/PR(GDPcap) only).</td></tr>
<tr class="pyaesa-optional-row" style="color:#c45f00;"><td style="vertical-align:top; white-space:nowrap;"><code>agg_reg</code></td><td>If <code>True</code>, reclassify MRIO regions with the <code>agg_reg_&lt;agg_version&gt;.csv</code> MRIO aggregation and disaggregation mapping. The mapping can keep native labels, aggregate several native regions into one target label, or disaggregate one native region across several target labels when a <code>weight</code> column is provided.<br>&#10;<strong>Default</strong> <code>False</code> keeps native source regions.</td></tr>
<tr class="pyaesa-optional-row" style="color:#c45f00;"><td style="vertical-align:top; white-space:nowrap;"><code>agg_sec</code></td><td>If <code>True</code>, reclassify MRIO sectors with the <code>agg_sec_&lt;agg_version&gt;.csv</code> MRIO aggregation and disaggregation mapping. The mapping can keep native labels, aggregate several native sectors into one target label, or disaggregate one native sector across several target labels when a <code>weight</code> column is provided.<br>&#10;<strong>Default</strong> <code>False</code> keeps native source sectors.</td></tr>
<tr class="pyaesa-optional-row" style="color:#c45f00;"><td style="vertical-align:top; white-space:nowrap;"><code>agg_version</code></td><td>Name token used to resolve the matching <code>agg_reg_&lt;agg_version&gt;.csv</code> and/or <code>agg_sec_&lt;agg_version&gt;.csv</code> MRIO aggregation and disaggregation mapping files in <code>data_raw/mrio/&lt;source&gt;/aggregation</code>. Required when <code>agg_reg</code> or <code>agg_sec</code> is True.<br>&#10;<strong>Defaults to</strong> an empty string for native source classification. Use the same token in downstream calls that should reuse the processed classification. If a mapping file has a <code>weight</code> column, weights must sum to <code>1</code> for each original label.</td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>years</code></td><td>Studied years. Accepts a single year, list, or range. If<br>&#10;<strong>omitted</strong>, all available MRIO<br>&#10;years for the selected source and <code>agg_version</code> are used.</td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>fu_code</code></td><td>Required functional unit code (for example <code>&quot;L1.a&quot;</code>,<br>&#10;<code>&quot;L2.c.b&quot;</code>). See<br>&#10;<code>data_raw/methodological_notes/methodological_note__asocc_fus_allocation_methods.pdf</code><br>&#10;for all available functional unit codes and the system<br>&#10;boundaries each represents.</td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>s_p</code></td><td>Producing sector filter(s), single string or list. If this is a<br>&#10;required axis for <code>fu_code</code> and the argument is <strong>omitted</strong>, the run<br>&#10;expands to all valid producing sectors. To identify valid sector<br>&#10;names, see the first column of the relevant<br>&#10;<code>data_raw/mrio/.../aggregation/.../agg_sec_template.csv</code> file. For<br>&#10;EXIOBASE sector definitions, see<br>&#10;<code>data_raw/mrio/exiobase_3/sector_classification.xlsx</code>; EXIOBASE<br>&#10;ixi and pxp use different sector lists.</td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>r_p</code></td><td>Producing region filter(s), single string or list. If this is a<br>&#10;required axis for <code>fu_code</code> and the argument is <strong>omitted</strong>, the run<br>&#10;expands to all valid producing regions. To identify valid region<br>&#10;names, see the first column of the relevant<br>&#10;<code>data_raw/mrio/.../aggregation/agg_reg_template.csv</code> file.</td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>r_c</code></td><td>Consuming region filter(s), single string or list. If this is a<br>&#10;required axis for <code>fu_code</code> and the argument is <strong>omitted</strong>, the run<br>&#10;expands to all valid consuming regions. To identify valid region<br>&#10;names, see the first column of the relevant<br>&#10;<code>data_raw/mrio/.../aggregation/agg_reg_template.csv</code> file.</td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>r_f</code></td><td>Final demand region filter(s), single string or list. If this is<br>&#10;a required axis for <code>fu_code</code> and the argument is <strong>omitted</strong>, the<br>&#10;run expands to all valid final demand regions. To identify valid<br>&#10;region names, see the first column of the relevant<br>&#10;<code>data_raw/mrio/.../aggregation/agg_reg_template.csv</code> file.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>group_indices</code></td><td>Whether multiple selected region or sector filter values are kept as separate result rows or summed into one result row after the function calculation has been performed.<br>&#10;&bull;&nbsp;<code>False</code> (<strong>default</strong>): keep selected values as independent rows.<br>&#10;&bull;&nbsp;<code>True</code>: sum selected values into one result row.<br>&#10;The function refuses to run when <code>group_indices=True</code> is used with <code>L2.a.b</code>, <code>L2.b.b</code>, or <code>L2.c.b</code> because summing output rows for CBA total demand boundaries can double count. For these functional units, change the upstream MRIO aggregation and disaggregation scope with <code>agg_reg</code>, <code>agg_sec</code>, and <code>agg_version</code> before running the study.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>method_plan</code></td><td><code>method_plan</code> <strong>defaults to</strong> <code>&quot;default&quot;</code> and accepts<br>&#10;<code>&quot;default&quot;</code>, <code>&quot;one_step&quot;</code>, <code>&quot;two_steps&quot;</code>, <code>&quot;pairs&quot;</code>, or<br>&#10;<code>&quot;one_step_pairs&quot;</code>. <strong>When omitted</strong>, all <span style="color:#366e9c"><strong>py</strong></span><span style="color:#c83737"><strong>aesa</strong></span> allocation methods<br>&#10;available for the selected <code>fu_code</code> are applied. See<br>&#10;<code>data_raw/methodological_notes/methodological_note__asocc_fus_allocation_methods.pdf</code><br>&#10;for the allocation methods available per functional unit,<br>&#10;including definitions and mathematical expressions.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>l1_methods</code></td><td>Optional L1 subset. <strong>Omit</strong> it to keep all L1 methods allowed<br>&#10;by <code>method_plan</code>. In <code>&quot;default&quot;</code>, this filters only L1 weights<br>&#10;used by two step methods. In <code>&quot;two_steps&quot;</code>, it filters the two<br>&#10;step cartesian L1 side.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>one_step_methods</code></td><td>Optional one step L2 subset. <strong>Omit</strong> it to keep all one<br>&#10;step methods allowed by <code>method_plan</code>.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>two_step_methods</code></td><td>Optional two step L2 subset. <strong>Omit</strong> it to keep all two<br>&#10;step L2 methods allowed by <code>method_plan</code>.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>l1_l2_pairs</code></td><td>Explicit pair list formatted as <code>&quot;L1METHOD::L2METHOD&quot;</code>.<br>&#10;<strong>Omit</strong> it unless <code>method_plan</code> is <code>&quot;pairs&quot;</code> or<br>&#10;<code>&quot;one_step_pairs&quot;</code>.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>l1_reg_aggreg</code></td><td>L1 aggregation mode for methods where timing matters<br>&#10;(<code>PR(GDPcap)</code>, <code>PR-HR(Ecap)</code> and <code>AR(Ecap)</code>).<br>&#10;&#160;<br>&#10;&bull;&nbsp;<code>&quot;pre&quot;</code>: aggregate regions before L1 computation.<br>&#10;&bull;&nbsp;<code>&quot;post&quot;</code> (<strong>default</strong>): compute on original regions, then<br>&#10;  aggregate.</td></tr>
<tr class="pyaesa-optional-row" style="color:#c45f00;"><td style="vertical-align:top; white-space:nowrap;"><code>lcia_method</code></td><td>LCIA method(s) selected for LCIA based allocation<br>&#10;methods (acquired rights (AR) methods at L1 and L2 and historical<br>&#10;responsibility (PR-HR) at L1). Options are for example<br>&#10;<code>&quot;pb_lcia&quot;</code> or <code>[&quot;pb_lcia&quot;, &quot;gwp100_lcia&quot;]</code>. <code>None</code> skips<br>&#10;LCIA characterization and excludes LCIA based allocation methods.<br>&#10;<strong>Defaults to</strong> <code>None</code>. <span style="color:#366e9c"><strong>py</strong></span><span style="color:#c83737"><strong>aesa</strong></span> currently supports LCIA based<br>&#10;allocation methods only for EXIOBASE sources. To add a custom<br>&#10;LCIA method with which run <code>process_mrio(...)</code>, follow<br>&#10;<code>README_add_custom_lcia_characterization_matrices.txt</code> in<br>&#10;<code>data_raw/mrio/exiobase_3/lcia/characterization_factors_matrices/</code><br>&#10;and pass the custom method file stem here.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>reference_years</code></td><td>Acquired rights (AR) methods reference year selector.<br>&#10;Accepts a single year, list, or range. <strong>If omitted</strong>, AR routes use<br>&#10;all historical years in the studied range up to the source registry<br>&#10;historical cutoff. For EXIOBASE 3.10.2 and OECD ICIO v2025, the<br>&#10;cutoff is 2022; other supported MRIO sources use their own<br>&#10;registry cutoff.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>ssp_scenario</code></td><td>SSP scenario name or list. <strong>Defaults to</strong><br>&#10;<code>[&quot;SSP1&quot;, &quot;SSP2&quot;, &quot;SSP3&quot;, &quot;SSP4&quot;, &quot;SSP5&quot;]</code> and is applied<br>&#10;when scenario dependent inputs are required.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>projection_mode</code></td><td>Projection policy for post historical years of L2<br>&#10;utilitarian (UT) methods (MRIO economic enacting metrics).<br>&#10;<strong>Defaults to</strong> <code>&quot;regression&quot;</code>.<br>&#10;&#160;<br>&#10;&bull;&nbsp;<code>&quot;regression&quot;</code>: project UT inputs for future years.<br>&#10;&bull;&nbsp;<code>&quot;historical_reuse&quot;</code>: reuse historical UT structures.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>reg_window</code></td><td>Historical regression fit window for regression mode.<br>&#10;Provide it as <code>range(start_year, end_year + 1)</code> or as an<br>&#10;explicit list of consecutive years in fit window order. When<br>&#10;<strong>omitted</strong>, the source registry supplies the <strong>default</strong> fit window from<br>&#10;the modeled year minimum through the source historical cutoff. For<br>&#10;EXIOBASE 3.10.2 and OECD ICIO v2025, this resolves to 1995 to<br>&#10;2022; other supported MRIO sources use their own registry window.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>l2_reuse_years</code></td><td>Historical L2 reuse year selector used by all UT<br>&#10;historical reuse routes. In <code>projection_mode=&quot;historical_reuse&quot;</code><br>&#10;it applies to all UT methods; in <code>projection_mode=&quot;regression&quot;</code><br>&#10;it applies to adjusted UT routes (<code>UT(FDa)</code>, <code>UT(GVAa)</code>),<br>&#10;which always use historical reuse as regression is not supported<br>&#10;(would require regression on full MRIO). <strong>If omitted</strong>, <strong>defaults to</strong><br>&#10;<code>reg_window</code> when required.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>output_format</code></td><td>Persisted output file format: <code>&quot;csv&quot;</code> (<strong>default</strong>),<br>&#10;<code>&quot;pickle&quot;</code>, or <code>&quot;parquet&quot;</code>.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>intermediate_outputs</code></td><td>Whether to write intermediate output families.<br>&#10;These outputs are for user audit and method inspection only; they<br>&#10;are not used by downstream package functions.<br>&#10;&#160;<br>&#10;&bull;&nbsp;<code>False</code> (<strong>default</strong>): skip writing enacting metrics and<br>&#10;  <code>utility_propagation_contrib</code> (L2*b for FUs) outputs.<br>&#10;&bull;&nbsp;<code>True</code>: write all output families.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>figures</code></td><td>Whether to render figures.<br>&#10;<strong>Default is</strong> <code>True</code>.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>figure_format</code></td><td>Figure render settings mapping. <strong>Defaults to</strong><br>&#10;<code>{&quot;format&quot;: &quot;png&quot;, &quot;dpi&quot;: 500}</code>.<br>&#10;&#160;<br>&#10;Nested keys:<br>&#10;&#160;<br>&#10;&bull;&nbsp;<code>format</code>: Figure file format. Accepted values are <code>&quot;png&quot;</code>,<br>&#10;  <code>&quot;pdf&quot;</code>, and <code>&quot;svg&quot;</code>.<br>&#10;&bull;&nbsp;<code>dpi</code>: Positive integer figure resolution used for raster<br>&#10;  outputs.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>figure_options</code></td><td>Figure product selector mapping. <strong>Defaults to</strong><br>&#10;<code>{&quot;per_method&quot;: True, &quot;multi_method&quot;: True}</code>.<br>&#10;&#160;<br>&#10;Nested keys:<br>&#10;&#160;<br>&#10;&bull;&nbsp;<code>per_method</code>: Whether to render method specific figures, with one separate<br>&#10;  figure for each allocation method.<br>&#10;&#160;<br>&#10;&bull;&nbsp;<code>multi_method</code>: Whether to render cross method comparison figures, with<br>&#10;  multiple allocation methods shown in the same figure.</td></tr>
<tr class="pyaesa-optional-row" style="color:#c45f00;"><td style="vertical-align:top; white-space:nowrap;"><code>figure_external_method</code></td><td>Optional external deterministic aSoCC selector<br>&#10;block used only for figure rendering. Use<br>&#10;<code>prepare_external_inputs(...)</code> to import the external aSoCC runnable CSV examples and <code>README_external_asocc_templates.txt</code>, then follow that README for<br>&#10;method syntax and data input format. This argument is valid only<br>&#10;when <code>figures=True</code>. <strong>Omit</strong> it to render only native<br>&#10;deterministic aSoCC method rows. <strong>Defaults to</strong> <code>None</code>.</td></tr>
<tr><td colspan="2" style="height:0.75rem;"></td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>refresh</code></td><td>If <code>True</code>, remove and rebuild the resolved deterministic aSoCC source and version output scope for this project, source label, and aggregate version. The cleared scope is the source and version <code>deterministic</code> folder under <code>&lt;project&gt;/B1_asocc/&lt;source_or_source__agg_version&gt;</code>. Processed MRIO inputs, processed population and GDP, raw downloads, and downstream aCC or ASR outputs are not refreshed. <strong>Defaults to</strong> <code>False</code>.</td></tr>
</tbody>
</table>

</details>

### Deterministic aSoCC with defaults where omitted

In [ ]:
from pyaesa import deterministic_asocc

deterministic_asocc(
    project_name=project_name_,
    source=source_,
    years=study_period_,
    fu_code=fu_code_,
    s_p=s_p_,
    r_c=r_c_,
    lcia_method=lcia_method_,
    refresh=True,
)

### Deterministic aSoCC with a given allocation method selected instead of all available methods

In [ ]:
deterministic_asocc(
    project_name=project_name_,
    source=source_,
    years=study_period_,
    fu_code=fu_code_,
    s_p=s_p_,
    r_c=r_c_,
    lcia_method=lcia_method_,
    method_plan="pairs",
    l1_l2_pairs=["EG(Pop)::UT(FDa)", "PR(GDPcap)::UT(FDa)", "AR(Ecap^{CBA_FD})::UT(FDa)"],
    refresh=True,
)

### Deterministic aSoCC with a subset of SSP scenarios

Using SSP scenarios for aSoCC means that the user is interested in prospective allocation. The study period is therefore extended to 2030 for the sake of this example.

In [ ]:
deterministic_asocc(
    project_name=f"{project_name_}_prosp",
    source=source_,
    years=range(study_period_[0], 2031),
    fu_code=fu_code_,
    s_p=s_p_,
    r_c=r_c_,
    lcia_method=lcia_method_,
    ssp_scenario=["SSP1", "SSP2"],
)

# What to do next

**Beginners**

If you are a new user in the process of discovering <span style="color:#366e9c"><strong>py</strong></span><span style="color:#c83737"><strong>aesa</strong></span>, it is recommend that you first discover all study objectives with the **basic features** available.
Have a look at the other notebooks available at [tutorials/study_objectives/0_study_objectives.md](../0_study_objectives.md)

**Experts**

If you are already familiar with <span style="color:#366e9c"><strong>py</strong></span><span style="color:#c83737"><strong>aesa</strong></span> and if you want to discover **advanced features** available, check out examples in the next section of this tutorial !

# Advanced features

Advanced features currently available include:
- Custom external aSoCC
- Custom LCIA method
- indices aggregation

### Custom external aSoCC

The package ships working external aSoCC examples for tests, the same way it
ships working external LCA examples. Import those runnable examples into a
project with the optional workflow
[tutorials/optional_workflows/external_asocc_lca_input_staging.ipynb](../../optional_workflows/external_asocc_lca_input_staging.ipynb).
That workflow writes the external aSoCC and external LCA example CSVs plus a
README explaining external method names, selector syntax, and deterministic or
Monte Carlo row preparation.

For deterministic aSoCC, external methods are used for figure comparison with
`figure_external_method`. They are not written to deterministic output tables,
because `deterministic_asocc(...)` does not compute external aSoCC result rows.
For uncertainty runs, the same staged external method names are selected with
the `external_method` argument.

In [ ]:
deterministic_asocc(
    project_name=f"{project_name_}_external_figures",
    source=source_,
    years=study_period_,
    fu_code=fu_code_,
    s_p=s_p_,
    r_c=r_c_,
    lcia_method=lcia_method_,
    figure_external_method={
        "one_step_methods": ["CO(S)"],
        "l1_l2_pairs": ["AR(E)::UT(S)"],
    },
)

### Indices aggregation

`group_indices=True` reports multiple selected region or sector indices as one
summed result row after the selected aSoCC scope is computed. The default
`False` keeps selected indices as separate rows.

The function refuses to run when `group_indices=True` is used with `L2.a.b`,
`L2.b.b`, or `L2.c.b`, because summing output rows for CBA total demand boundaries
can double count. For those functional units, define aggregated regions or sectors
during MRIO processing with `agg_reg`, `agg_sec`, and `agg_version`.

The basic notebook example uses `L2.c.b`, so this aggregation example switches
to `L2.c.a`.

In [ ]:
deterministic_asocc(
    project_name=f"{project_name_}_group_indices",
    source=source_,
    years=study_period_,
    fu_code="L2.c.a",
    s_p=["Paper"],
    r_c=["FR", "DE"],
    lcia_method=lcia_method_,
    group_indices=True,
)

### Custom LCIA method

To
add a custom LCIA method, follow
`data_raw/mrio/exiobase_3/lcia/characterization_factors_matrices/README_add_custom_lcia_characterization_matrices.txt`
and run `process_mrio(...)` with the custom method file stem.